## Vidore v3

### Imports

In [4]:
import torch
from PIL import Image
from datasets import load_dataset
from colpali_engine.models import ColQwen2, ColQwen2Processor
from transformers import AutoModel, AutoTokenizer
from sentence_transformers import SentenceTransformer

# --- CONFIGURATION ---
DATASET_NAME = "vidore/vidore_v3_physics" 
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

### Load dataset from HF

In [5]:
# --- LOAD DATASET ---
ds_corpus = load_dataset(DATASET_NAME, 'corpus', split='test')
ds_queries = load_dataset(DATASET_NAME, 'queries', split='test')
ds_qrels = load_dataset(DATASET_NAME, 'qrels', split='test')
ds_metadata = load_dataset(DATASET_NAME, 'documents_metadata', split='test')

Generating test split: 100%|██████████| 42/42 [00:00<00:00, 13774.40 examples/s]


### Examples

Pdf documents

In [12]:
num_pdfs = len(ds_metadata) # Number of documents in the corpus
print(f"Number of pdf documents in the corpus of {DATASET_NAME}: {num_pdfs}")

n_examples = 3
print(f"\nSample metadata for first {n_examples} documents:")
for i in range(n_examples):
    print(f'{i+1}')
    print(f"filename : {ds_metadata[i]['file_name']}")
    print(f"doc_id   : {ds_metadata[i]['doc_id']}")
    print(f"url      : {ds_metadata[i]['url']}")


Number of pdf documents in the corpus of vidore/vidore_v3_physics: 42

Sample metadata for first 3 documents:
1
filename : Autrement_Ch-0-Presentation_cours_autrement.pdf
doc_id   : Autrement_Ch-0-Presentation_cours_autrement
url      : https://un-peu-de-physique.fr/wp-content/uploads/2021/01/Ch-0-Presentation_cours_autrement.pdf
2
filename : Autrement_Ch-1a-La-quete-de-lunite-La-methode-scientifique.pdf
doc_id   : Autrement_Ch-1a-La-quete-de-lunite-La-methode-scientifique
url      : https://un-peu-de-physique.fr/wp-content/uploads/2023/11/Ch-1a-La-quete-de-lunite-La-methode-scientifique.pdf
3
filename : Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023.pdf
doc_id   : Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023
url      : https://un-peu-de-physique.fr/wp-content/uploads/2023/11/Ch-1b-La-quete-de-lunite-Etat-en-2023.pdf


Pages across all the documents

In [7]:
num_documents = len(ds_corpus)
print(f"Number of pdf pages in the corpus of {DATASET_NAME}: {num_documents}")

print(f"\nSample data for first {n_examples} pages in the corpus:")

for i in range(n_examples):
    print(f'{i+1}')
    print(f"doc_id             : {ds_corpus[i]['doc_id']}")
    print(f"markdown           : {ds_corpus[i]['markdown'][:30]}...")
    print(f"page_number_in_doc : {ds_corpus[i]['page_number_in_doc']}")


# Each element of ds_corpus is a page from a pdf document, with its corresponding markdown transcription
# The pdf document can be retrieve from ds_metadata using the 'doc_id' field

ind = ds_metadata['doc_id'].index(ds_corpus[0]['doc_id'])  # Get the index of the document metadata for the first page in the corpus
print(ds_metadata[ind])  # Print the metadata of the document

Number of pdf pages in the corpus of vidore/vidore_v3_physics: 1674

Sample data for first 3 pages in the corpus:
1
doc_id             : Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023
markdown           : Un peu de Science pour compren...
page_number_in_doc : 0
2
doc_id             : Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023
markdown           : La quête de l'Unité* (fin) 2 L...
page_number_in_doc : 1
3
doc_id             : Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023
markdown           : Relativité restreinte 11 Temps...
page_number_in_doc : 10
{'file_name': 'Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023.pdf', 'doc_id': 'Autrement_Ch-1b-La-quete-de-lunite-Etat-en-2023', 'url': 'https://un-peu-de-physique.fr/wp-content/uploads/2023/11/Ch-1b-La-quete-de-lunite-Etat-en-2023.pdf', 'doc_type': 'slides', 'doc_language': 'french', 'doc_year': '2023', 'visual_types': ['chart', 'infographic', 'image', 'table', 'text'], 'page_number': 64, 'license': 'CC-BY-4.0'}


Queries

In [17]:
num_queries = len(ds_queries)

languages = ['french', 'english', 'spanish', 'german', 'italian', 'portuguese']
index_between_languages = num_queries // len(languages)

print(f"Number of queries in {DATASET_NAME}: {num_queries}")

print(f"\nSample data for first {n_examples} queries:")

# Random queries
n_examples = 1
for i in range(n_examples):

    print('=================================')
    for l in languages:
        print(f'{i+1} ({l})')
        print(f"Query  : {ds_queries[i+index_between_languages*languages.index(l)]['query']}")
        print(f"Answer : {ds_queries[i+index_between_languages*languages.index(l)]['raw_answers'][0]}")

Number of queries in vidore/vidore_v3_physics: 1812

Sample data for first 1 queries:
1 (french)
Query  : Quelle est la distance caractéristique à laquelle la force nucléaire forte résiduelle devient fortement attractive entre les nucléons ?
Answer : La force nucléaire forte résiduelle, responsable de la cohésion des nucléons au sein du noyau atomique, devient fortement attractive à une distance caractéristique d’environ 1 à 2 femtomètres (fm), où 1 fm=10−15 m. À des distances inférieures à environ 0,5 fm, la force devient répulsive, empêchant les nucléons de s’effondrer complètement. Son intensité maximale d’attraction se situe vers 1 fm, ce qui correspond approximativement à la taille typique d’un nucléon. Au-delà de 2 à 3 fm, la force décroît très rapidement et devient négligeable comparée à la répulsion électrostatique entre protons chargés positivement.
1 (english)
Query  : What is the characteristic distance at which the residual strong nuclear force becomes strongly attractive b

Query-doc relation

In [41]:
num_query_doc_relation = len(ds_qrels)
print(f"Number of query-document relations in {DATASET_NAME}: {num_query_doc_relation}")

n_examples = 3
print(f"\nSample data for first {n_examples} query-document relations:")
for i in range(n_examples):
    print('=================================')
    print(f'{i+1}')

    query_id = ds_qrels[i]['query_id']
    corpus_id = ds_qrels[i]['corpus_id']

    # Print the question and answer
    questions = [
        ds_queries[query_id + j*index_between_languages]['query'] for j in range(len(languages))]
    answers = [
        ds_queries[query_id + j*index_between_languages]['raw_answers'][0] for j in range(len(languages))
    ]

    
    print(f'question    : {questions[0]}')
    print(f'            : {questions[1]}')
    print(f'            : {questions[2]}')
    print(f'            : {questions[3]}')
    print(f'            : {questions[4]}')
    print(f'            : {questions[5]}')
  
    print(f'answer      : {answers[0]}')

    print()

    # Get the corresponding document page
    document_page = ds_corpus[corpus_id]
    pdf_url = ds_metadata[ds_metadata['doc_id'].index(document_page['doc_id'])]['url']
    markdown = document_page['markdown']
    page_in_doc = document_page['page_number_in_doc']

    print(f'page number : {page_in_doc}')
    print(f'markdown    : {markdown[:100]}...')
    print(f'page URL    : {pdf_url}')


Number of query-document relations in vidore/vidore_v3_physics: 13068

Sample data for first 3 query-document relations:
1
question    : Quelle est la distance caractéristique à laquelle la force nucléaire forte résiduelle devient fortement attractive entre les nucléons ?
            : What is the characteristic distance at which the residual strong nuclear force becomes strongly attractive between nucleons?
            : ¿Cuál es la distancia característica a la que la fuerza nuclear fuerte residual se vuelve fuertemente atractiva entre los nucleones?
            : Qual è la distanza caratteristica alla quale la forza nucleare forte residua diventa fortemente attrattiva tra i nucleoni?
            : Welche ist die charakteristische Distanz, bei der die starke, residuelle Kernkraft zwischen den Nukleonen stark anziehend wird?
            : Qual é a distância característica na qual a força nuclear forte residual se torna fortemente atrativa entre os nucleons?
answer      : La force nucl

In [10]:
# Check if queries have a language field
print("Checking language distribution in queries:")
print(f"Query fields: {ds_queries.features}")
print()

# Check if there's a language field
if 'language' in ds_queries.features:
    languages = set([q['language'] for q in ds_queries])
    print(f"Languages in queries: {languages}")
    for lang in languages:
        count = sum(1 for q in ds_queries if q['language'] == lang)
        print(f"  {lang}: {count} queries")
else:
    print("No 'language' field found in queries")
    
print("\n" + "="*50)
print("Checking first few queries in detail:")
for i in range(3):
    print(f"\nQuery {i+1}:")
    print(ds_queries[i])

Checking language distribution in queries:
Query fields: {'query_id': Value('int64'), 'query': Value('string'), 'language': Value('string'), 'query_types': List(Value('string')), 'query_format': Value('string'), 'content_type': List(Value('string')), 'raw_answers': List(Value('string')), 'query_generator': Value('string'), 'query_generation_pipeline': Value('string'), 'source_type': Value('string'), 'query_type_for_generation': Value('string'), 'answer': Value('string')}

Languages in queries: {'german', 'english', 'french', 'italian', 'portuguese', 'spanish'}
  german: 302 queries
  english: 302 queries
  french: 302 queries
  italian: 302 queries
  portuguese: 302 queries
  spanish: 302 queries

Checking first few queries in detail:

Query 1:
{'query_id': 0, 'query': 'Quelle est la distance caractéristique à laquelle la force nucléaire forte résiduelle devient fortement attractive entre les nucléons ?', 'language': 'french', 'query_types': ['extractive', 'numerical'], 'query_format':

In [ ]:
num_query_doc_relation = len(ds_qrels)
print(f"Number of query-document relations in {DATASET_NAME}: {num_query_doc_relation}")

list_queries = [ds_queries[ds_qrels[i]['query_id']]['query'] for i in range(num_query_doc_relation)]


# print(f'{len(list_queries)=}')



Number of query-document relations in vidore/vidore_v3_physics: 13068
list_queries[0]='Quelle est la distance caractéristique à laquelle la force nucléaire forte résiduelle devient fortement attractive entre les nucléons ?'


In [37]:
import numpy as np
print(f'{len(list_queries)=}')
print(f'{len(np.unique(list_queries))=}')

len(list_queries)=13068
len(np.unique(list_queries))=1812


In [40]:
ds_qrels[0]

{'query_id': 0,
 'corpus_id': 1315,
 'score': 2,
 'content_type': ['Text', 'Text'],
 'bounding_boxes': [{'annotator': 0,
   'x1': 1020,
   'x2': 2089,
   'y1': 419,
   'y2': 510},
  {'annotator': 0, 'x1': 1025, 'x2': 2171, 'y1': 587, 'y2': 732}]}